In [92]:
from openadmet.toolkit.database.chembl import PPBChEMBLCurator
from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey
from tqdm.auto import tqdm
tqdm.pandas()
import datamol as dm
import os
import subprocess

# Curating PPB data from ChEMBL and pushing to a remote intake catalog

Our goal is to curate activity data from ChEMBL and push this to a remote location with a catalog that can be used by others to look up our data. This will enable consistency and rapid dissemination of our work as well as an over-time evolution of our data curation practices. 

We use the `Intake` package for a lightweight self-describing data parsing workflow. Read more about intake here: https://intake.readthedocs.io/en/latest/index.html


Here we gather `PPB` data permissivley from ChEMBL (ie without activity based curation). Here we curate for BAO BAO_0000366 (cell free) and no target.

We then aggregate  measurements on the same compound by taking the mean and median. This is the most basic form of curation available, but serves as a good baseline for our initial models. 



## gather ChEMBL data

First we need to gather in our data from ChEMBL using our SQL API defined in `openadmet-toolkit`

We use `OPENADMET_CANONICAL_SMILES` and `OPENADMET_INCHIKEY` to distinguish our ML ready representation from the source SMILES

In [93]:
def gather_chembl_data_PPB(chembl_ver: int, organism: str = None):
    print(f"working on target (organism={organism})")
    pctc = PPBChEMBLCurator(version=chembl_ver, organism=organism)
    activity_data = pctc.get_activity_data(return_as="df")
    print("canonicalising raw data")
    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data

In [94]:
chembl_ver = 35

In [57]:
from openadmet.toolkit.webservices.credentials import S3Settings
from openadmet.toolkit.webservices.s3 import S3Bucket

# Setup S3

After curating our data we would like to push to a remote bucket to save both the raw data and the catalog

In [5]:
os.environ["AWS_ACCESS_KEY_ID"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_access_key_id"],
    text=True
).strip()

os.environ["AWS_SECRET_ACCESS_KEY"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_secret_access_key"],
    text=True
).strip()

In [59]:
settings = S3Settings()

In [60]:
bucket = "openadmet-data-public-dev"

In [61]:
bucket = S3Bucket.from_settings(settings, bucket)

In [62]:
import datetime

In [63]:
t = datetime.datetime.now()

In [64]:
date = t.strftime("%Y-%m-%d")

In [65]:
location=f"ChEMBL{chembl_ver}_PPB"

In [66]:
import os
from pathlib import Path

location_path = Path(location)

In [67]:
location_path.mkdir(exist_ok=True)

Gather PPB data separately for each species (Human and Mouse): `HPPB`/`MPPB` 

In [ ]:
species = {
    "HPPB": "Homo sapiens",
    "MPPB": "Mus musculus",
}

uris_raw = {}
uris_agg = {}

for target, organism in species.items():
    agg, raw = gather_chembl_data_PPB(chembl_ver, organism=organism)

    fname_agg = f"ChEMBL_PPB_{target}_aggregated.parquet"
    fname_raw = f"ChEMBL_PPB_{target}_raw.parquet"

    agg.reset_index(drop=True).to_parquet(location_path/fname_agg, index=False)
    raw.reset_index(drop=True).to_parquet(location_path/fname_raw, index=False)

    bucket_destination_agg = location + "/" + fname_agg
    bucket.push_file(location_path/fname_agg, bucket_destination_agg)
    bucket_destination_raw = location + "/" + fname_raw
    bucket.push_file(location_path/fname_raw, bucket_destination_raw)

    uris_agg[target] = bucket.to_uri(bucket_destination_agg)
    uris_raw[target] = bucket.to_uri(bucket_destination_raw)

# Build the Intake Catalog

We have sucessfully aggregted our data and pushed it to a remote destination. Now for others to consume our data, we are going to make an `Intake` catalog such that our data can be readily made available. 

The workflow here is drawn from the `creator` walkthrough from the main intake tutorials https://intake.readthedocs.io/en/latest/walkthrough2.html

TODO: add descriptions to the catalog

In [113]:
import intake

In [ ]:
intake.Catalog?

In [115]:
cat = intake.entry.Catalog()

In [ ]:
uris_agg

In [ ]:
uris_raw

In [118]:
for k,v in uris_agg.items():
    cat[k+"_aggregated"] = intake.readers.PandasParquet(v)

In [119]:
for k,v in uris_raw.items():
    cat[k+"_raw"] = intake.readers.PandasParquet(v)

## Push the Catalog

Ok now we have made the catalog, lets push it to the remote location so it can live alongside the data. 

The catalog can then be used from S3 or from github etc, anything that exposes a file-like API. 

In [ ]:
cat

In [121]:
catname = f"CATALOG_{location}.yaml"

In [122]:
cat.to_yaml_file(catname)

In [123]:
cat_location = location+ "/" +catname

In [ ]:
cat_location

In [125]:
bucket.push_file(catname, cat_location)

In [126]:
cat_uri = bucket.to_uri(cat_location)

In [127]:
# Now can read the catalog from URI
# cat = intake.Catalog.from_yaml_file("s3://openadmet-data-public-dev/ChEMBL34_permissive_2025-02-12/CATALOG_ChEMBL34_permissive_2025-02-12.yaml")

## Trimming data
Converts aggregated % protein bound → % unbound → log10(% unbound), then trims outliers to NaN (row-safe).

In [128]:
import pandas as pd

import numpy as np

def clip_and_log_transform(y, multiplier=1.0):
    y = np.clip(y * multiplier, a_min=1e-9, a_max=None)
    return np.log10(y)

def trim_outliers_to_nan(df, column, lower=None, upper=None,
                         lower_quantile=0.01, upper_quantile=0.99):
    mask = df[column].notna()
    if lower is None:
        lower = df.loc[mask, column].quantile(lower_quantile)
    if upper is None:
        upper = df.loc[mask, column].quantile(upper_quantile)
    trim_mask = mask & ((df[column] < lower) | (df[column] > upper))
    df_out = df.copy()
    df_out.loc[trim_mask, column] = np.nan
    return df_out, {'lower': lower, 'upper': upper,
                    'trimmed': int(trim_mask.sum()),
                    'remaining': int(df_out[column].notna().sum())}

In [130]:
uris_trimmed = {}

for target, uri in uris_agg.items():
    target_agg = cat[f"{target}_aggregated"].read()

    # Convert % bound → % unbound → log10(% unbound)
    target_agg["LogUnbound"] = target_agg["standard_value_mean"].apply(
        lambda x: clip_and_log_transform(100 - x, multiplier=1.0) if pd.notna(x) else x
    )

    df_trimmed, summary = trim_outliers_to_nan(
        target_agg,
        column="LogUnbound",
        lower=-2,
        upper=2,
    )
    # keep raw column consistent with log column
    df_trimmed.loc[df_trimmed["LogUnbound"].isna(), "standard_value_mean"] = np.nan

    print(
        f"[{target}] Trimmed {summary['trimmed']} LogUnbound values outside "
        f"[{summary['lower']}, {summary['upper']}] to NaN. "
        f"Remaining labeled values: {summary['remaining']}"
    )

    fname_trimmed = f"trimmed/ChEMBL_PPB_{target}_aggregated.parquet"
    Path(location_path/fname_trimmed).parent.mkdir(exist_ok=True, parents=True)
    df_trimmed.reset_index(drop=True).to_parquet(location_path/fname_trimmed, index=False)

    bucket_destination_trimmed = location + "/" + fname_trimmed
    bucket.push_file(location_path/fname_trimmed, bucket_destination_trimmed)
    uris_trimmed[target] = bucket.to_uri(bucket_destination_trimmed)

[HPPB] Trimmed 62 LogUnbound values outside [-2, 2] to NaN. Remaining labeled values: 4499
[MPPB] Trimmed 15 LogUnbound values outside [-2, 2] to NaN. Remaining labeled values: 1421


In [131]:
for k, v in uris_trimmed.items():
    cat[k + "_trimmed_aggregated"] = intake.readers.PandasParquet(v)

# Re-push updated catalog with trimmed entries
cat.to_yaml_file(catname)
bucket.push_file(catname, cat_location)